# Petals to the Metal — ResNet50 迁移学习（PyTorch + TPU）

本 Notebook 使用 **torchvision 预训练 ResNet50** 做迁移学习，涵盖：
1. 用 TensorFlow 读取 TFRecord 竞赛数据
2. 加载 ImageNet 预训练 ResNet50，替换分类头
3. 在 Kaggle TPU 上微调 104 类花卉分类模型
4. 推理测试集并生成 `submission.csv`

**适合人群**：想用预训练模型 + TPU 快速打比赛的同学。

## 1. 环境准备

Kaggle TPU 环境预装了 PyTorch、TensorFlow 和 `torch_xla`，通常不需要额外安装。

In [ ]:
# torch_xla 仅在 TPU 环境可用，缺失则回退 GPU/CPU
# Kaggle TPU VM 默认已预装，无需手动 pip install

## 2. 导入库 & TPU 初始化

In [ ]:
import io, time
from pathlib import Path

import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ── TPU 检测 ─────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.distributed.xla_multiprocessing as xmp

    devices = xm.get_xla_supported_devices()
    if devices:
        device = xm.xla_device()
        tpu_available = True
        print(f'TPU available: {len(devices)} cores')
    else:
        raise RuntimeError('No TPU devices')
except Exception:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tpu_available = False
    print(f'TPU not available, using: {device}')
    if device.type == 'cuda':
        print(f'  GPU: {torch.cuda.get_device_name(0)}')

# 全局常量
NUM_CLASSES = 104
IMAGE_SIZE  = 224
BATCH_SIZE  = 64             # 单 core 的 batch
EPOCHS      = 30             # 迁移学习不需要太多轮
LR          = 1e-4           # 预训练模型用更低的学习率

if tpu_available:
    BATCH_SIZE = 32           # TPU 显存较小，ResNet50 需要更小 batch
    print(f'  TPU batch size adjusted to {BATCH_SIZE}')

## 3. 数据集：TensorFlow 读取 TFRecord

Kaggle 官方数据是 TFRecord 格式（JPEG 编码的图片 + 标签）。我们用 TF 解析，PyTorch Dataset 封装。

**为什么用 TensorFlow 而不是 `tfrecord` 包？**
—— Kaggle 环境预装了 TF，零额外依赖。

In [ ]:
class PetalsDataset(Dataset):
    """用 TensorFlow 解析 TFRecord，封装为 PyTorch Dataset。

    数据目录结构::

        data/
        └── tfrecords-jpeg-224x224/
            ├── train/   (16 个 .tfrec 文件)
            ├── val/     (16 个 .tfrec 文件)
            └── test/    (16 个 .tfrec 文件)
    """

    def __init__(self, data_dir, image_size=224, split='train', transform=None):
        if split not in ('train', 'val', 'test'):
            raise ValueError(f"split must be 'train', 'val', or 'test', got '{split}'")
        self.split = split
        self.transform = transform
        self.samples = []

        # 1. 找到所有 .tfrec 文件
        tfrecord_dir = Path(data_dir) / f'tfrecords-jpeg-{image_size}x{image_size}' / split
        tfrecord_paths = sorted(tfrecord_dir.glob('*.tfrec'))
        if not tfrecord_paths:
            raise FileNotFoundError(f"No .tfrec files in '{tfrecord_dir}'")

        # 2. 定义 TFRecord schema（test 集没有 class 字段）
        if split == 'test':
            feature_desc = {
                'image': tf.io.FixedLenFeature([], tf.string),
                'id':    tf.io.FixedLenFeature([], tf.string),
            }
        else:
            feature_desc = {
                'image': tf.io.FixedLenFeature([], tf.string),
                'class': tf.io.FixedLenFeature([], tf.int64),
            }

        # 3. 读取全部记录到内存
        raw_ds = tf.data.TFRecordDataset([str(p) for p in tfrecord_paths])
        for record in raw_ds:
            parsed = tf.io.parse_single_example(record, feature_desc)
            sample = {'image': parsed['image'].numpy()}         # JPEG 字节
            if split == 'test':
                sample['id'] = parsed['id'].numpy()
            else:
                sample['class'] = parsed['class'].numpy()      # int64 标签
            self.samples.append(sample)
        print(f'  {split}: {len(self.samples)} 张图片')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        r = self.samples[idx]
        # JPEG byte → RGB PIL Image
        img = Image.open(io.BytesIO(r['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.split == 'test':
            img_id = r['id'].decode('utf-8') if isinstance(r['id'], bytes) else r['id']
            return img, img_id
        return img, int(r['class'])

## 4. 模型：ResNet50（ImageNet 预训练）

ResNet50 的核心理念是 **残差连接**（skip connection）—— 将输入直接加到输出上，解决了深层网络的梯度消失问题。

**架构概览**：

```
Input 3×224×224
  │
  ▼ Conv 7×7, stride=2  →  64×112×112
  │
  ▼ MaxPool 3×3, stride=2  →  64×56×56
  │
  ▼ Bottleneck ×3  (layer1)  →  256×56×56
  │
  ▼ Bottleneck ×4  (layer2)  →  512×28×28
  │
  ▼ Bottleneck ×6  (layer3)  →  1024×14×14
  │
  ▼ Bottleneck ×3  (layer4)  →  2048×7×7
  │
  ▼ GlobalAvgPool  →  2048
  │
  ▼ FC(2048→104)  →  输出 104 类
```

每个 Bottleneck 包含三层卷积（1×1 → 3×3 → 1×1）+ BatchNorm + ReLU，输入通过跳跃连接加到输出。
约 **2560 万参数**，ImageNet Top-1 准确率 ~80%。

**迁移学习策略**：
- 加载 ImageNet 预训练权重（`ResNet50_Weights.DEFAULT`）
- 替换最后的全连接层 `fc`（1000 → 104 类）
- 用较低学习率（1e-4）微调全部层

In [ ]:
def build_resnet50(num_classes=104, pretrained=True):
    """构建 ResNet50 模型（可选 ImageNet 预训练权重）。"""
    weights = tv_models.ResNet50_Weights.DEFAULT if pretrained else None
    model = tv_models.resnet50(weights=weights)
    model.fc = nn.Linear(2048, num_classes)
    return model


# 快速验证输入输出尺寸
_model = build_resnet50(NUM_CLASSES)
_x = torch.randn(1, 3, 224, 224)
_y = _model(_x)
print(f'Input:   {tuple(_x.shape)}')
print(f'Output:  {tuple(_y.shape)}')
print(f'Params:  {sum(p.numel() for p in _model.parameters()):,}')
del _model, _x, _y  # 释放显存

## 5. 训练引擎

包含：累加器、准确率计算、训练一轮、验证、完整训练循环。

TPU 训练的关键差异：
- `xm.optimizer_step(optimizer)` 代替 `optimizer.step()`
- `MpDeviceLoader` 包装 DataLoader 以预取数据到 TPU

In [ ]:
class Accumulator:
    """累加器 —— 方便在多个 batch 上累加 loss 和准确率。"""
    def __init__(self, n):
        self.data = [0.0] * n
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    def __getitem__(self, i):
        return self.data[i]


def accuracy(y_hat, y):
    """计算预测正确的数量。"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)
    return float((y_hat.type(y.dtype) == y).type(y.dtype).sum())


def evaluate(net, data_iter, eval_device):
    """在验证集上评估准确率。"""
    net.eval()
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(eval_device), y.to(eval_device)
            y_hat = net(X)
            metric.add(accuracy(y_hat, y), y.numel())
    return metric[0] / metric[1]


def train_one_epoch(net, loader, loss_fn, optimizer, train_device):
    """训练一个 epoch。TPU 时用 ``xm.optimizer_step``。

    返回 (avg_loss, accuracy)。
    """
    net.train()
    metric = Accumulator(3)

    for X, y in loader:
        X, y = X.to(train_device), y.to(train_device)
        optimizer.zero_grad()

        y_hat = net(X)
        l = loss_fn(y_hat, y)
        l.mean().backward()

        if tpu_available:
            xm.optimizer_step(optimizer)           # ← TPU 专用
        else:
            optimizer.step()                       # ← GPU / CPU

        metric.add(l.sum().item() * y.numel(), accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2], metric[1] / metric[2]


def train_model(net, train_iter, val_iter, num_epochs, lr):
    """完整训练流程：TPU / GPU / CPU 自适应。

    返回: (train_losses, train_accs, val_accs, best_val_acc)
    """
    net.to(device)

    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs)
    loss_fn = nn.CrossEntropyLoss()

    # TPU 需要用 MpDeviceLoader 预取数据
    if tpu_available:
        train_loader = pl.MpDeviceLoader(train_iter, device)
        val_loader   = pl.MpDeviceLoader(val_iter, device)
    else:
        train_loader = train_iter
        val_loader   = val_iter

    train_losses, train_accs, val_accs = [], [], []
    best_acc = 0.0
    t0_total = time.time()

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        train_l, train_acc = train_one_epoch(
            net, train_loader, loss_fn, optimizer, device)
        val_acc = evaluate(net, val_loader, device)
        scheduler.step()

        train_losses.append(train_l)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(net.state_dict(), 'best_model.pth')

        elapsed = time.time() - t0
        eta = (elapsed / epoch) * (num_epochs - epoch)
        eta_str = f' | ETA: {eta/60:.0f}min' if eta > 60 else f' | ETA: {eta:.0f}s'

        if epoch % 5 == 0 or epoch == 1:
            print(f'epoch {epoch:3d}/{num_epochs} | '
                  f'train loss {train_l:.4f}, train acc {train_acc:.4f}, '
                  f'val acc {val_acc:.4f} | '
                  f'{elapsed:.0f}s/epoch{eta_str}')

    total = time.time() - t0_total
    print(f'\n训练完成！总用时: {total/60:.1f}min  |  最佳 val acc: {best_acc:.4f}')
    return train_losses, train_accs, val_accs, best_acc

## 6. 数据增强 & 加载

ResNet50 使用与 GoogLeNet 相同的预处理：ImageNet 均值/标准差归一化。

In [ ]:
DATA_DIR = '/kaggle/input/tpu-getting-started'

# ── 训练 transform（带增强）──
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

# ── 验证 transform（不做增强）──
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## 7. 训练！

In [ ]:
model = build_resnet50(num_classes=NUM_CLASSES, pretrained=True)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

# 下面这行开始训练。TPU v3-8 上约 40~60 分钟（ResNet50 比 GoogLeNet 大）
train_losses, train_accs, val_accs, best_acc = train_model(
    model, train_iter, val_iter,
    num_epochs=EPOCHS, lr=LR,
)

## 8. 推理 & 生成提交文件

In [ ]:
# 加载最佳模型权重
model.load_state_dict(torch.load('best_model.pth',
                        map_location=torch.device('cpu')))
model.to(device).eval()

# 测试集（不做增强）
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('正在加载测试集...')
test_ds = PetalsDataset(DATA_DIR, IMAGE_SIZE, 'test', test_tf)
test_iter = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
print(f'测试集: {len(test_ds)} 张')

ids_all, preds_all = [], []
with torch.no_grad():
    for imgs, img_ids in test_iter:
        logits = model(imgs.to(device))
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        ids_all.extend(img_ids)

print(f'预测完成: {len(ids_all)} 条')

# 写入 submission.csv
with open('/kaggle/working/submission.csv', 'w') as f:
    f.write('id,label\n')
    for img_id, pred in zip(ids_all, preds_all):
        f.write(f'{img_id},{pred}\n')
print('/kaggle/working/submission.csv 已保存！')

## 9. 总结

### 与 GoogLeNet 的对比

| 维度 | GoogLeNet | ResNet50 |
|------|-----------|----------|
| 参数量 | 7.1M | 25.6M |
| 训练方式 | 从零训练 | ImageNet 预训练 + 微调 |
| 收敛速度 | 慢（60 轮 ~75%） | 快（30 轮 ~90%+） |
| 学习率 | 1e-3 | 1e-4 |
| 架构特点 | Inception 多分支 | Bottleneck 残差连接 |
| BatchNorm | 每层后 | 每层后（原生） |

### 关键设计决策

| 决策 | 选择 | 原因 |
|------|------|------|
| 模型 | ResNet50 (25.6M 参数) | ImageNet 预训练，迁移效果好 |
| 优化器 | AdamW + CosineAnnealing | 自适应学习率 + 权重衰减 |
| 学习率 | 1e-4 | 预训练模型不需要大学习率 |
| 数据增强 | RandomCrop + HFlip + Rotation | 抑制过拟合 |
| ColorJitter | **不用** | 花卉分类颜色是关键特征 |
| Epochs | 30 | 迁移学习收敛快，太多会过拟合 |

### 预期效果

- 验证准确率：30 轮约 **88~94%**（ImageNet 预训练迁移）
- 比赛前排约 95%+（EfficientNet / ConvNeXt）

### 改进方向

- 换成 ResNet101 / ResNet152（更多参数，更好的特征）
- 换成 EfficientNet / ConvNeXt（更新的 backbone）
- 加 MixUp / CutMix 增广（减小过拟合）
- 分层学习率（底层小 LR，分类头大 LR）